# Curating COVID example data


Canadian HV.1 and JN.1 data was downloaded on 2026-01-14 from [ViruSeq Portal](https://virusseq-dataportal.ca/explorer?filters=%7B%22content%22%3A%5B%7B%22content%22%3A%7B%22fieldName%22%3A%22analysis.first_published_at%22%2C%22value%22%3A1696132800000%7D%2C%22op%22%3A%22%3E%3D%22%7D%2C%7B%22content%22%3A%7B%22fieldName%22%3A%22analysis.first_published_at%22%2C%22value%22%3A1704085199999%7D%2C%22op%22%3A%22%3C%3D%22%7D%2C%7B%22content%22%3A%7B%22fieldName%22%3A%22analysis.lineage_analysis.lineage_name%22%2C%22value%22%3A%5B%22HV.1%22%2C%22JN.1%22%2C%22JN.1.1%22%2C%22JN.1.10%22%2C%22JN.1.18%22%2C%22JN.1.19%22%2C%22JN.1.2%22%2C%22JN.1.22%22%2C%22JN.1.31%22%2C%22JN.1.39%22%2C%22JN.1.4%22%2C%22JN.1.4.5%22%2C%22JN.1.4.7%22%2C%22JN.1.41%22%2C%22JN.1.44%22%2C%22JN.1.45%22%2C%22JN.1.47.1%22%2C%22JN.1.5%22%2C%22JN.1.52%22%2C%22JN.1.55%22%2C%22JN.1.6%22%2C%22JN.1.60%22%2C%22JN.1.8%22%2C%22JN.1.8.3%22%2C%22JN.1.9%22%5D%7D%2C%22op%22%3A%22in%22%7D%5D%2C%22op%22%3A%22and%22%7D) by selecting all data submitted between 2023-10-01 and 2023-12-31 with lineage name equal to 'HV.1', 'JN.1' or starting with 'JN.1.'.

Sequence used for reference in aligning was MN908947.3 downloaded from https://www.ncbi.nlm.nih.gov/nuccore/MN908947.3?report=fasta on 2026-01-14.


Import necessary packages


In [1]:
import pandas as pd
import Bio.SeqIO
from tqdm import tqdm
import yaml

## Double checking data contains only HV.1 and JN.1 sequences
### Metadata

In [8]:
metadata = pd.read_csv('HV_1_and_JN_1_metadata.csv', parse_dates=[ 'submission date', 'sample collection date'])
with open('sub_variants_map.yaml', 'r') as file:
    sub_variants_map = yaml.safe_load(file)
allowed_lineages = [item for sublist in sub_variants_map.values() for item in sublist]
metadata = metadata[metadata['lineage name'].isin(allowed_lineages)]
metadata.to_csv('HV_1_and_JN_1_metadata.csv', index=False)

### Sequences


In [7]:
HV_1_and_JN1_sequences = []
for record in tqdm(Bio.SeqIO.parse('HV_1_and_JN_1_sequences.fasta', 'fasta')):
    if record.id in allowed_lineages:
        HV_1_and_JN1_sequences.append(record)
with open('HV_1_and_JN_1_sequences.fasta', 'w') as file:
    Bio.SeqIO.write(HV_1_and_JN1_sequences, file, 'fasta')
file.close()

6745it [00:00, 19650.63it/s]


## Explore metadata

Counts of lineages

In [ ]:
metadata['lineage name'].value_counts()

How many JN.1 & all its sublineages

In [ ]:
len(metadata[metadata['lineage name']!='HV.1'])

Collection date data for all HV.1.

In [ ]:
metadata[metadata['lineage name']=='HV.1']['sample collection date'].describe()

Collection date data for all JN.1 and its sublineages.

In [ ]:
metadata[metadata['lineage name']!='HV.1']['sample collection date'].describe()

## Aligning and masking Sequences

Sequences will be aligned using Nextclade.

### Obtaining Reference genome

In [ ]:
%%bash
source activate beast_pype_data_curation
nextclade dataset get --name 'nextstrain/sars-cov-2/wuhan-hu-1/orfs' --output-dir 'nextclade-sars-cov-2'

### Align Sequences

In [ ]:
%%bash
source activate beast_pype_data_curation
nextclade run \
   --input-dataset nextclade-sars-cov-2 \
   --output-all=output/ \
   virusseq-search-export/sequences.fasta
mv output/nextclade.aligned.fasta HV_1_and_JN_1_sequences.fasta

## Putting JN.1 Sequences in there own dataset

In [ ]:
JN1_ids = metadata[metadata['lineage name']!='HV.1']['fasta header name'].to_list()
HV1_and_JN1_sequences = []
JN1_sequences = []

for record in tqdm(Bio.SeqIO.parse('HV_1_and_JN_1_sequences.fasta', 'fasta')):
    if record.id in JN1_ids:
        JN1_sequences.append(record)
with open('JN_1_sequences.fasta', 'w') as file:
    Bio.SeqIO.write(JN1_sequences, file, 'fasta')
file.close()

## Saving the metadata for HV_1_and_JN_1_sequences.fasta and JN_1_sequences.fasta

In [ ]:
metadata[metadata['lineage name']!='HV.1'].to_csv('JN_1_metadata.csv', index=False)
metadata.to_csv('HV_1_and_JN_1_metadata.csv', index=False)

In [ ]:
sorted(metadata['lineage name'].unique().tolist())

In [ ]:
metadata.columns